# EDA — proprete.csv
Score de Vivabilité · Signalements de propreté urbaine à Paris (DansMaRue)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

## 1. Chargement

In [ ]:
FILE = 'architecture-data/brute/score_de_vivabilité/proprete.csv'

df = pd.read_csv(FILE, sep=None, engine='python')
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
df['DATE DECLARATION'] = pd.to_datetime(df['DATE DECLARATION'], errors='coerce')

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 2. Infos générales

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Valeurs manquantes — critique pour ce dataset

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, max(3, len(df.columns) * 0.45)))
colors = ['#D85A30' if v > 50 else '#BA7517' if v > 20 else '#1D9E75' for v in missing.values]
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel('% manquant')
ax.set_title('Valeurs manquantes par colonne (rouge > 50%, orange > 20%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.axvline(50, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

display(missing.rename('% manquant').to_frame())

## 4. Doublons

In [ ]:
n_dup = df.duplicated().sum()
print(f'Doublons : {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    display(df[df.duplicated(keep=False)].head(10))

## 5. Détection colonnes clés

In [ ]:
def find_col(df, *keywords):
    for kw in keywords:
        match = [c for c in df.columns if kw.lower() in c.lower()]
        if match: return match[0]
    return None

col_map = {
    'id'           : find_col(df, 'id declaration', 'id_dmr', 'identifiant'),
    'type'         : find_col(df, 'type declaration', 'type decl'),
    'sous_type'    : find_col(df, 'sous type', 'sous_type'),
    'date'         : find_col(df, 'date declaration', 'date_decl'),
    'arrondissement': find_col(df, 'arrondissement', 'arrond'),
    'geo_point'    : find_col(df, 'geo_point', 'geopoint', 'coord'),
    'mois'         : find_col(df, 'mois declaration', 'mois_decl'),
    'annee'        : find_col(df, 'annee declaration', 'annee_decl'),
}

print('Colonnes détectées :')
for k, v in col_map.items():
    status = '✅' if v else '❌'
    print(f'  {status}  {k:20s} → {v}')
print(f'\nColonnes non mappées : {[c for c in df.columns if c not in col_map.values()]}')

## 6. Répartition par type de déclaration

In [ ]:
type_col = col_map['type']
if type_col:
    vc_type = df[type_col].value_counts()
    colors = ['#378ADD','#1D9E75','#D85A30','#BA7517','#888780','#534AB7','#D4537E','#2E8B57','#4B0082','#DC143C']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.barh(vc_type.index, vc_type.values, color=colors[:len(vc_type)])
    ax1.set_title('Signalements par type')
    ax1.set_xlabel('Nombre de signalements')
    ax1.invert_yaxis()

    ax2.pie(vc_type.values, labels=vc_type.index, autopct='%1.1f%%',
            colors=colors[:len(vc_type)], startangle=140)
    ax2.set_title('Part par type (%)')

    plt.suptitle('Distribution par type de déclaration', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    display(vc_type.rename('count').to_frame())

## 7. Répartition par arrondissement

In [ ]:
arr_col = col_map['arrondissement']
if arr_col:
    print(f'Complétude arrondissement : {df[arr_col].notna().sum()}/{len(df)} ({df[arr_col].notna().mean()*100:.1f}%)')
    vc_arr = df[arr_col].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(vc_arr.index.astype(str), vc_arr.values, color='#1D9E75')
    ax.set_title('Signalements par arrondissement parisien')
    ax.set_xlabel('Arrondissement')
    ax.set_ylabel('Nombre de signalements')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('❌ Colonne arrondissement non trouvée')

## 8. Évolution temporelle des signalements

In [ ]:
date_col = col_map['date']
if date_col and pd.api.types.is_datetime64_any_dtype(df[date_col]):
    df['mois_period'] = df[date_col].dt.to_period('M')
    ts = df.groupby('mois_period').size()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts.index.astype(str), ts.values, color='#378ADD', marker='o', markersize=3)
    ax.set_title('Volume mensuel de signalements')
    ax.set_xlabel('Mois')
    ax.set_ylabel('Nb signalements')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 9. Cardinalité des colonnes catégorielles

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
cardinality = pd.DataFrame({
    'colonne'    : cat_cols,
    'unique'     : [df[c].nunique() for c in cat_cols],
    'exemple'    : [str(df[c].dropna().iloc[0])[:50] if not df[c].dropna().empty else 'N/A' for c in cat_cols],
    '% manquant' : [round(df[c].isnull().mean() * 100, 1) for c in cat_cols],
}).sort_values('unique', ascending=False)

display(cardinality)

## 10. Résumé + plan Silver

In [ ]:
print('=' * 55)
print('RÉSUMÉ EDA — proprete.csv')
print('=' * 55)
print(f'  Lignes              : {len(df):,}')
print(f'  Colonnes            : {len(df.columns)}')
print(f'  Doublons            : {df.duplicated().sum()}')
missing_count = (df.isnull().sum() > 0).sum()
print(f'  Cols avec NaN       : {missing_count}')
if col_map['arrondissement']: print(f'  Arrondissements     : {df[col_map["arrondissement"]].nunique()}')
if col_map['type']:           print(f'  Types déclaration   : {df[col_map["type"]].nunique()}')
if col_map['sous_type']:      print(f'  Sous-types          : {df[col_map["sous_type"]].nunique()}')
if col_map['date']:           print(f'  Période             : {df[col_map["date"]].min().date()} → {df[col_map["date"]].max().date()}')
print('=' * 55)
print()
print('ACTIONS SILVER REQUISES :')
print('  → Parser geo_point_2d ("lat, lon") en deux colonnes float')
print('  → Corriger CODE POSTAL aberrant 75248 → 75020')
print('  → Agréger par arrondissement + mois pour volumétrie')
print('  → Score propreté = nb signalements / population arrondissement')
print('  → Pondérer par gravité (encombrants > graffitis > autre)')
print('  → Export Parquet pour Gold scoring vivabilité')